# 02 事件研究：异常收益率与累计异常收益率

本 Notebook 使用事件研究法，计算加息和降息事件下各市场的 AR 与 CAR，并对平均 CAR 进行显著性检验。

In [ ]:
from pathlib import Path

import sys

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns



PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name != 'T2_FED_world_stock_market':

    PROJECT_DIR = Path.cwd() / 'homework' / 'Topics_01' / 'T2_FED_world_stock_market' if (Path.cwd() / 'homework' / 'Topics_01' / 'T2_FED_world_stock_market').exists() else Path.cwd()

CODE_DIR = PROJECT_DIR / 'code'

if str(CODE_DIR) not in sys.path:

    sys.path.append(str(CODE_DIR))



from data_utils import MARKET_CONFIG

from event_study_utils import build_market_event_panel, average_car_by_type, final_car_ttest, asymmetry_test



plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']

plt.rcParams['axes.unicode_minus'] = False

sns.set_style('whitegrid')



DATA_CLEAN = PROJECT_DIR / 'data_clean'

OUTPUT_DIR = PROJECT_DIR / 'output'



returns_df = pd.read_csv(DATA_CLEAN / 'global_index_returns.csv', index_col=0, parse_dates=True)

events_df = pd.read_csv(DATA_CLEAN / 'fed_events_aligned.csv', parse_dates=['date'])

for key in MARKET_CONFIG:

    col = f'{key}_event_date'

    events_df[col] = pd.to_datetime(events_df[col])



returns_df.head()


## 1. 构建事件面板

In [ ]:
panel_list = []
for key in MARKET_CONFIG:
    panel = build_market_event_panel(returns_df, events_df, key, left=5, right=5, est_left=30, est_right=6)
    panel_list.append(panel)

event_panel = pd.concat(panel_list, ignore_index=True)
event_panel['market_label'] = event_panel['market_key'].map({k: v['label'] for k, v in MARKET_CONFIG.items()})
display(event_panel.head())
print('事件面板记录数:', len(event_panel))

In [ ]:
avg_car = average_car_by_type(event_panel)
avg_car['market_label'] = avg_car['market_key'].map({k: v['label'] for k, v in MARKET_CONFIG.items()})
display(avg_car.head(20))

## 2. 绘制各市场平均 CAR

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex=True, sharey=True)
axes = axes.flatten()
palette = {'hike': 'firebrick', 'cut': 'navy'}

for idx, key in enumerate(MARKET_CONFIG):
    ax = axes[idx]
    market_df = avg_car[avg_car['market_key'] == key]
    for event_type in ['hike', 'cut']:
        sub = market_df[market_df['type'] == event_type]
        if sub.empty:
            continue
        ax.plot(sub['event_day'], sub['avg_car'], marker='o', linewidth=1.8, color=palette[event_type], label='加息' if event_type == 'hike' else '降息')
    ax.axvline(0, color='black', linestyle='--', alpha=0.6)
    ax.axhline(0, color='gray', linestyle=':', alpha=0.6)
    ax.set_title(MARKET_CONFIG[key]['label'])
    ax.set_xlabel('事件日')
    ax.set_ylabel('平均 CAR')
    ax.legend()

plt.suptitle('不同市场在加息与降息事件下的平均 CAR 路径', y=1.02, fontsize=15)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'average_car_facet.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. 平均 CAR 显著性检验

In [ ]:
ttest_df = final_car_ttest(event_panel)
ttest_df['market_label'] = ttest_df['market_key'].map({k: v['label'] for k, v in MARKET_CONFIG.items()})
ttest_df = ttest_df[['market_key', 'market_label', 'type', 'mean_car', 'std_car', 'n_events', 't_stat', 'p_value']]
display(ttest_df.sort_values(['type', 'market_key']))

In [ ]:
final_car = event_panel[event_panel['event_day'] == 5].copy()
summary_table = final_car.groupby(['market_key', 'type']).agg(
    mean_car=('car', 'mean'),
    median_car=('car', 'median'),
    n_events=('event_id', 'nunique')
).reset_index()
summary_table['market_label'] = summary_table['market_key'].map({k: v['label'] for k, v in MARKET_CONFIG.items()})
display(summary_table)

## 4. 非对称性检验（选做）

In [ ]:
asym_results = []
for key in MARKET_CONFIG:
    result = asymmetry_test(final_car, key)
    if result is not None:
        result['market_label'] = MARKET_CONFIG[key]['label']
        asym_results.append(result)

asym_df = pd.DataFrame(asym_results)
display(asym_df)

## 5. 保存结果与结论

In [ ]:
event_panel.to_csv(DATA_CLEAN / 'event_study_panel.csv', index=False)
avg_car.to_csv(OUTPUT_DIR / 'average_car_by_market.csv', index=False)
ttest_df.to_csv(OUTPUT_DIR / 'car_ttest_results.csv', index=False)
summary_table.to_csv(OUTPUT_DIR / 'event_window_car_summary.csv', index=False)
if not asym_df.empty:
    asym_df.to_csv(OUTPUT_DIR / 'asymmetry_test_results.csv', index=False)
print('事件研究结果已保存')

- 若加息组 CAR 在多数市场为负且显著，说明紧缩冲击在短期内压制权益市场表现。
- 若降息组 CAR 更显著为正，则表明流动性宽松更容易带来短期估值修复。
- 不同市场间 CAR 差异，通常与资本流动开放程度、美元流动性依赖度以及本币汇率制度相关。